# ForgeAI public-dev local evaluation — GPU execution gate

Accelerator: **4 x NVIDIA L4**. Internet: **Off**. Public competition data only.
No submission or training. Review README and pass laptop controls first.
Packages must already match source-pins.json, prepared separately using the official starter
installation procedure. This notebook neither installs nor silently repairs the environment.
A frozen, reviewed subset and a previously reconciled candidate package are mandatory.
Full traces and verification logs stay outside Git and are never fed back to the agent.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Attach this entire local_eval directory as a private tooling input.
TOOLS = Path('/kaggle/input/datasets/REPLACE/local-eval/local_eval')
CONFIG = {
    "candidate": "p1",
    "subset": "smoke-5",
    "authorization": "",  # AUTHORIZE FORGEAI GEMMA4 LOCAL EVAL p1 ON smoke-5
    "gpu_authorized": False,  # Explicit approval to consume 4x L4 quota.
    "internet_off_confirmed": False,
    "confirmation_only": False,  # True only for a separately approved confirm-20 run.
    "winner_evidence_sha256": "",  # Required for confirm-20, from reviewed dev comparison.
    "package_reconciled": False,  # Require local scope + portable compile reconciliation.
    "package": "/kaggle/input/datasets/REPLACE/candidate/submission.zip.bin",
    "manifest": "/kaggle/input/datasets/REPLACE/candidate/manifest.json",
    "manifest_sha256": "",  # Trusted raw-file pin from reviewed reconciliation evidence.
    "receipt_sha256": "",
    "subset_file": str(TOOLS / "subsets.json"),
    "subset_sha256": "627f17c74f81450a5ff1787175a30ab9af7ebd9588697ef9e1bbe29367920316",  # Pin frozen before any model run.
    "data": "/kaggle/input/competitions/gemma-4-developer-agent",
    "wheelhouse": "/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse",
    "model_path": "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2",
    "output": "/kaggle/working/forgeai-local-eval-p1-smoke5-attempt1",
    "wall_seconds": 7200,  # Includes worker imports/server start/setup/tasks; +10s cleanup grace.
}
expected = f"AUTHORIZE FORGEAI GEMMA4 LOCAL EVAL {CONFIG['candidate']} ON {CONFIG['subset']}"
if CONFIG['authorization'] != expected or CONFIG['gpu_authorized'] is not True:
    raise RuntimeError('STOP: explicit candidate/subset GPU authorization required')
if not CONFIG['internet_off_confirmed'] or not CONFIG['package_reconciled']:
    raise RuntimeError('STOP: Internet Off and reconciled package required')
if not (TOOLS / 'runtime.py').is_file():
    raise RuntimeError('Attach reviewed local_eval tooling directory')
CONFIG_PATH = Path('/kaggle/working/local-eval-config.json')
with CONFIG_PATH.open('x', encoding='utf-8') as stream:
    json.dump(CONFIG, stream, indent=2)
print('Configuration recorded; next cell starts GPU serving and official evaluation.')


In [ ]:
# Entire serving/evaluation worker is bounded by the parent watchdog.
subprocess.run([sys.executable, '-B', str(TOOLS / 'runtime.py'), str(CONFIG_PATH)], check=True)


Download the complete output directory, including run.json, tasks.jsonl,
events.jsonl, completion.json, worker.log and official/. Preserve failed/partial attempts too.
Analyze on the laptop using analysis.py. Stop the Kaggle session after downloading;
allocated GPU session time, including idle/setup, may consume quota.
No ZIP submission is produced and no Kaggle submission action is present.
